# Chapter 8: Tree-Based Algorithms and Ensemble Methods

## Ringkasan Bab & Penjelasan Teoretis
Algoritma berbasis pohon mempartisi ruang fitur secara rekursif ke dalam wilayah-wilayah persegi panjang berdasarkan aturan pemisahan yang meminimalkan ketidakmurnian simpul (*node impurity*). Metode *ensemble* menggabungkan banyak pohon keputusan untuk menghasilkan model yang lebih akurat dan tahan terhadap *overfitting*.

### 1. Decision Trees, Random Forests (Bagging), dan Gradient Boosting
* **Decision Trees (`DecisionTreeClassifier`)**: Memilih fitur dan ambang batas pemisahan terbaik menggunakan kriteria **Gini Impurity** ($G = 1 - \sum p_k^2$) atau **Entropy / Information Gain** ($H = -\sum p_k \log_2 p_k$). Tidak memerlukan penyekalaan fitur, namun rentan mengalami *overfitting* jika kedalaman pohon (`max_depth`) tidak dibatasi.
* **Random Forests & Bagging (`RandomForestClassifier`)**: Melatih ratusan pohon keputusan secara paralel pada sampel acak dengan pengembalian (*bootstrap samples*) dan subset fitur acak (`max_features`). Mengurangi varians secara drastis dibandingkan pohon tunggal.
* **Gradient Boosting (`GradientBoostingClassifier`)**: Melatih pohon keputusan secara berurutan (*sequential*), di mana setiap pohon baru dilatih untuk memprediksi kesalahan residu (*pseudo-residuals*) dari kumpulan pohon sebelumnya menggunakan laju pembelajaran (`learning_rate`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report

wine = load_wine()
X, y = wine.data, wine.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=2024)

# 1. Decision Tree Classifier
dt_clf = DecisionTreeClassifier(max_depth=4, random_state=2024)
dt_clf.fit(X_train, y_train)
y_pred_dt = dt_clf.predict(X_test)
print(f"Decision Tree Accuracy: {accuracy_score(y_test, y_pred_dt):.3f}")

plt.figure(figsize=(14, 7))
plot_tree(dt_clf, feature_names=wine.feature_names, class_names=wine.target_names, filled=True, rounded=True)
plt.title("Decision Tree Visualization (Wine Dataset)")
plt.show()

# 2. Random Forest Classifier (Bagging)
rf_clf = RandomForestClassifier(n_estimators=100, random_state=2024)
rf_clf.fit(X_train, y_train)
y_pred_rf = rf_clf.predict(X_test)
print(f"Random Forest Accuracy: {accuracy_score(y_test, y_pred_rf):.3f}")

# 3. Gradient Boosting Classifier
gb_clf = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=2024)
gb_clf.fit(X_train, y_train)
y_pred_gb = gb_clf.predict(X_test)
print(f"Gradient Boosting Accuracy: {accuracy_score(y_test, y_pred_gb):.3f}")

### 2. Hyperparameter Tuning & Perbandingan Metode Ensemble
Penyetelan hyperparameter seperti `n_estimators`, `max_depth`, `min_samples_split`, dan `learning_rate` mengoptimalkan keseimbangan antara kapasitas belajar dan generalisasi model *ensemble*.

In [ ]:
# 4. Hyperparameter Tuning pada Random Forest dengan GridSearchCV
param_grid_rf = {
    'n_estimators': [50, 100, 150],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5]
}
grid_rf = GridSearchCV(RandomForestClassifier(random_state=2024), param_grid_rf, cv=5, scoring='accuracy')
grid_rf.fit(X_train, y_train)
print("Best Random Forest Params:", grid_rf.best_params_)
print(f"Best RF CV Score: {grid_rf.best_score_:.3f}")

# 5. Perbandingan Feature Importances: Random Forest vs Gradient Boosting
importances_df = pd.DataFrame({
    'Feature': wine.feature_names,
    'Random Forest Importance': rf_clf.feature_importances_,
    'Gradient Boosting Importance': gb_clf.feature_importances_
}).sort_values('Random Forest Importance', ascending=False)

display(importances_df.head(10))

# Plot Perbandingan Akurasi Cross-Validation Ketiga Model
models = {
    'Decision Tree': dt_clf,
    'Random Forest': grid_rf.best_estimator_,
    'Gradient Boosting': gb_clf
}
cv_results = {name: cross_val_score(m, X, y, cv=5).mean() for name, m in models.items()}
plt.figure(figsize=(8, 5))
plt.bar(cv_results.keys(), cv_results.values(), color=['skyblue', 'royalblue', 'navy'])
plt.ylim(0.8, 1.0)
plt.ylabel('Mean 5-Fold CV Accuracy')
plt.title('Comparison of Tree-Based and Ensemble Methods')
plt.show()